# 02 Error and Input Diagnostics

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb002_evidence.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb002_evidence.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb002_followup.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb002_followup.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb003_input_sensitivity.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb003_input_sensitivity.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`render_cm_viridis.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb002_evidence.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'ad2473d37d7453201f54d9e6e7ca33bed3f6c3b8'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb002_followup.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a71354ef25382cc2f7a7d16955931064a996a4ee'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb003_input_sensitivity.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'bed42ae58ea76784e58a997cad4d8e8252b670f6'}, '05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'c233e0fead21ab2fe3f0977c0c0d1a4f4596d1fb'}}

## exp_cmb002_evidence

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb002_evidence.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb002_evidence.py)  
Git blob: `ad2473d37d7453201f54d9e6e7ca33bed3f6c3b8`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-002: deterministic, train-free, author-held-out error evidence.

This script analyses the completed EXP-CMB-001 models and 2,560 OOF predictions.
It must NEVER read original X_test/y_test, modify baseline files, train a model,
or infer a new deployable preprocessing rule. Outputs are research diagnostics.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb002_evidence.py
#!/usr/bin/env python3
"""EXP-CMB-002: deterministic, train-free, author-held-out error evidence.

This script analyses the completed EXP-CMB-001 models and 2,560 OOF predictions.
It must NEVER read original X_test/y_test, modify baseline files, train a model,
or infer a new deployable preprocessing rule. Outputs are research diagnostics.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import scipy
from scipy.ndimage import distance_transform_edt
import sklearn

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

ROOT = Path(__file__).resolve().parents[4]
DIR = ROOT / "05_model_optimize" / "MLP" / "Combined"
sys.path.insert(0, str(DIR / "scripts"))  # exact original Pipeline scaler class
import pixel_transform  # noqa:F401

LABELS = "0123456789ABCDEF"
WRITERS = (41, 42, 43, 44)
FOCUS = ("0", "D", "8", "B")
RIVAL = {"0": "D", "D": "0", "8": "B", "B": "8"}
FEATURES = ("cx", "cy", "width", "height", "ink_ratio", "gray_mean",
            "stroke_proxy")
NPZDIR = ROOT / "02_data" / "Hand_made"
BASE = DIR / "results" / "EXP_CMB_001"


def sha256(path):
    with open(path, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def write_csv(path, rows, headers=None):
    rows = list(rows)
    if headers is None:
        assert rows
        headers = list(rows[0])
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        wr = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        wr.writeheader()
        wr.writerows(rows)


def load_inputs():
    manifest = json.loads((BASE / "experiment_manifest.json").read_text(encoding="utf-8"))
    assert manifest["experiment_id"] == "EXP-CMB-001"
    assert sha256(ROOT / manifest["input_data_paths"]["official"]) == manifest["dataset_sha256"]["official_v3"]
    predpath = BASE / "OOF_predictions.csv"
    with open(predpath, newline="", encoding="utf-8-sig") as f:
        pred = list(csv.DictReader(f))
    assert len(pred) == 2560
    table = {}
    raw = {}
    labels = {}
    for w in WRITERS:
        path = NPZDIR / ("ml26_" + str(w) + ".npz")
        assert sha256(path) == manifest["dataset_sha256"]["handmade"][str(w)]
        with np.load(path, allow_pickle=False) as d:
            xx = d["img"].copy()
            yy = d["label_id"].astype(int)
            labeltext = d["label"].astype(str)
        assert xx.shape == (640, 28, 28) and xx.dtype == np.uint8
        assert np.array_equal(np.bincount(yy, minlength=16), np.full(16, 40))
        assert np.array_equal(labeltext, np.array(list(LABELS))[yy])
        raw[w], labels[w] = xx, yy
    for row in pred:
        w, idx = int(row["writer"]), int(row["index_in_writer"])
        key = (w, idx)
        assert w in WRITERS and 0 <= idx < 640 and key not in table
        assert row["true_label"] == LABELS[labels[w][idx]]
        row["writer"], row["index_in_writer"] = w, idx
        assert row["normal_correct"] == str(int(row["normal_pred"] == row["true_label"]))
        assert row["combined_correct"] == str(int(row["combined_pred"] == row["true_label"]))
        table[key] = row
    assert len(table) == 2560 and all((w, i) in table for w in WRITERS for i in range(640))
    assert sum(int(x["normal_correct"]) for x in pred) == 1602
    assert sum(int(x["combined_correct"]) for x in pred) == 2080
    return manifest, pred, table, raw, labels


def load_models(manifest):
    zpath = BASE / "trained_models.zip"
    models = {}
    with zipfile.ZipFile(zpath) as zf:
        for name, exp_sha in manifest["models_zip_members_sha256"].items():
            matches = [s for s in zf.namelist() if s.endswith("/" + name) or s == name]
            assert len(matches) == 1, (name, zf.namelist())
            data = zf.read(matches[0])
            assert hashlib.sha256(data).hexdigest() == exp_sha, name
            models[name] = joblib.load(io.BytesIO(data))
    assert len(models) == 5
    return models


def logits(pipe, x):
    net = pipe.named_steps["model"]
    assert net.activation == "relu" and len(net.coefs_) == 2
    xx = np.asarray(x, dtype=np.float64).reshape(-1, 784) / 255.0
    h = np.maximum(xx @ net.coefs_[0] + net.intercepts_[0], 0.0)
    return h @ net.coefs_[1] + net.intercepts_[1]


def geometry(image, threshold=32):
    im = np.asarray(image, dtype=float)
    mask = im >= threshold
    count = int(mask.sum())
    if count == 0:
        return dict(cx=np.nan, cy=np.nan, width=np.nan, height=np.nan,
                    ink_ratio=0.0, gray_mean=float(im.mean()),
                    foreground_mean=np.nan, ink_mass=0.0, stroke_proxy=np.nan,
                    bbox_aspect=np.nan, center_offset=np.nan)
    ys, xs = np.nonzero(mask)
    mass = float(im[mask].sum())
    cx = float((im * np.arange(28)[None, :]).sum() / max(float(im.sum()), 1e-12))
    cy = float((im * np.arange(28)[:, None]).sum() / max(float(im.sum()), 1e-12))
    width = int(xs.max() - xs.min() + 1)
    height = int(ys.max() - ys.min() + 1)
    # Same thresholded, intensity-weighted CoM used in Normal's Spatial Response.
    wt = np.where(mask, im, 0.0)
    cx = float((wt * np.arange(28)[None, :]).sum() / mass)
    cy = float((wt * np.arange(28)[:, None]).sum() / mass)
    dist = distance_transform_edt(mask)
    return dict(cx=cx, cy=cy, width=width, height=height,
                ink_ratio=count/784.0, gray_mean=float(im.mean()),
                foreground_mean=float(im[mask].mean()), ink_mass=mass,
                stroke_proxy=float((2.0*dist[mask]).mean()),
                bbox_aspect=width/max(height, 1),
                center_offset=float(np.hypot(cx-13.5, cy-13.5)))


def prepare_features(pred, raw):
    feats = []
    for row in pred:
        w, i = row["writer"], row["index_in_writer"]
        f = geometry(raw[w][i], 32)
        f64, f128 = geometry(raw[w][i], 64), geometry(raw[w][i], 128)
        f = {**{k: row[k] for k in ("writer", "index_in_writer", "true_label",
                                   "normal_pred", "combined_pred", "normal_correct",
                                   "combined_correct")},
             "transition": ("both_correct" if row["normal_correct"] == "1" and row["combined_correct"] == "1"
                          else "corrected" if row["normal_correct"] == "0" and row["combined_correct"] == "1"
                          else "new_error" if row["normal_correct"] == "1" else
                          "persistent_same_error" if row["normal_pred"] == row["combined_pred"]
                          else "persistent_changed_error"),
             **f}
        for t, g in ((64, f64), (128, f128)):
            for k in ("cx", "cy", "width", "height", "ink_ratio"):
                f[k + "_t" + str(t)] = g[k]
        feats.append(f)
    assert len(feats) == 2560
    return feats


def make_class_tables(pred, out):
    c = np.zeros((16, 16), dtype=int)
    writer_cms = {}
    transitions = Counter()
    pairwriter = Counter()
    for r in pred:
        t, n, p = r["true_label"], r["normal_pred"], r["combined_pred"]
        w = r["writer"]
        c[LABELS.index(t), LABELS.index(p)] += 1
        pairwriter[(w, t, p)] += 1
        transitions[("N" if t == n else "W", "C" if t == p else "W",
                     "same" if n == p else "different")] += 1
    assert int(np.trace(c)) == 2080 and c.sum() == 2560
    p, rec, f1, support = precision_recall_fscore_support(
        [r["true_label"] for r in pred], [r["combined_pred"] for r in pred],
        labels=list(LABELS), zero_division=0)
    write_csv(out / "per_class_metrics.csv", [
        {"class": t, "support":int(support[i]), "precision":p[i], "recall":rec[i],
         "f1":f1[i], "correct":int(c[i, i])} for i,t in enumerate(LABELS)])
    wrows = []
    for w in WRITERS:
        rr = [r for r in pred if r["writer"] == w]
        mat = confusion_matrix([r["true_label"] for r in rr],
                               [r["combined_pred"] for r in rr], labels=list(LABELS))
        assert mat.sum() == 640
        writer_cms[w] = mat
        wrows.append({"writer":w,"n":len(rr),
                      "accuracy":float(np.trace(mat)/len(rr)),
                      "macro_recall":float(np.mean(np.diag(mat)/40.0)),
                      **{(t+"_to_"+p):int(mat[LABELS.index(t), LABELS.index(p)])
                         for t,p in (("0","D"),("D","0"),("8","B"),("B","8"))}})
    write_csv(out/"writer_metrics.csv", wrows)
    transrows = []
    for t in LABELS:
        for n in LABELS:
            for pnew in LABELS:
                count = sum(1 for r in pred if r["true_label"]==t and
                            r["normal_pred"]==n and r["combined_pred"]==pnew)
                if count:
                    transrows.append({"actual":t,"normal_prediction":n,
                                      "combined_prediction":pnew, "count":count,
                                      "transition": "corrected" if n!=t and pnew==t else
                                        "new_error" if n==t and pnew!=t else
                                        "both_correct" if n==t and pnew==t else
                                        "persistent_same_error" if n==pnew else
                                        "persistent_changed_error"})
    write_csv(out/"error_transitions.csv",transrows)
    return c, writer_cms, wrows, transrows


def save_cms(cm, writer_cms, out):
    target = out / "figures" / "writer_confusion_matrices"
    target.mkdir(parents=True, exist_ok=True)
    cases = [("pooled", cm, 160)] + [
        ("writer_" + str(w), writer_cms[w], 40) for w in WRITERS]
    for name, mat, maximum in cases:
        for kind in ("count", "normalized"):
            val = mat if kind=="count" else 100.0*mat/np.maximum(mat.sum(axis=1,keepdims=True),1)
            fig,ax = plt.subplots(figsize=(9,8))
            im = ax.imshow(val, cmap="viridis", vmin=0,vmax=maximum if kind=="count" else 100)
            fig.colorbar(im, ax=ax, fraction=.045, pad=.03)
            ax.set(xticks=np.arange(16),yticks=np.arange(16),xticklabels=list(LABELS),
                   yticklabels=list(LABELS),xlabel="Predicted",ylabel="Actual",
                   title=name+" | "+kind+" | OOF")
            for i in range(16):
                for j in range(16):
                    if mat[i,j]:
                        ax.text(j,i,format(val[i,j],".1f") if kind=="normalized" else str(mat[i,j]),
                                ha="center",va="center",fontsize=6,
                                color="black" if val[i,j]>.55*(maximum if kind=="count" else 100) else "white")
            fig.tight_layout()
            fig.savefig(target/(name+"_"+kind+".png"),dpi=200)
            plt.close(fig)


def stratified_permutation(feats, out, repeats=2500):
    """Within each true-class and writer, shuffle the error label; BH across features."""
    rng = np.random.default_rng(20261116)
    rr = [r for r in feats if r["true_label"] in FOCUS]
    groups = defaultdict(list)
    for i,r in enumerate(rr):
        groups[(r["writer"],r["true_label"])].append(i)
    assert len(rr)==640 and len(groups)==16
    e = np.asarray([int(r["combined_correct"]=="0") for r in rr],dtype=float)
    X = np.asarray([[float(r[k]) for k in FEATURES] for r in rr])
    assert np.isfinite(X).all()
    Z = np.zeros_like(X)
    center_error = e.copy()
    for ix in groups.values():
        z = X[ix]
        Z[ix] = (z-z.mean(axis=0))/np.maximum(z.std(axis=0),1e-6)
        center_error[ix] -= e[ix].mean()
    observed = (Z.T@center_error)/len(rr)
    sims = np.zeros((repeats,len(FEATURES)))
    for b in range(repeats):
        pe=e.copy()
        for ix in groups.values():
            pe[ix]=rng.permutation(pe[ix])
        sims[b] = (Z.T @ (pe - (e-center_error)))/len(rr)
    pvals = (1+np.sum(np.abs(sims)>=np.abs(observed)[None,:],axis=0))/(repeats+1)
    # Benjamini-Hochberg q-values.
    order=np.argsort(pvals)
    qs=np.empty(len(pvals))
    running=1.0
    for rank_rev in range(len(pvals)-1,-1,-1):
        j=order[rank_rev]
        running=min(running,float(pvals[j])*len(pvals)/(rank_rev+1))
        qs[j]=running
    rows=[]
    for j,k in enumerate(FEATURES):
        low=Z[e==0,j]
        high=Z[e==1,j]
        rows.append({"feature":k,"n":len(rr),"n_errors":int(e.sum()),
                     "error_minus_correct_mean_z":float(high.mean()-low.mean()),
                     "centered_covariance":float(observed[j]),
                     "stratified_permutation_p":float(pvals[j]),
                     "bh_fdr_q":float(qs[j]),"permutations":repeats})
    write_csv(out/"feature_associations.csv",rows)
    groupsout=[]
    for (w,t),ix in sorted(groups.items()):
        actual = np.asarray(ix)
        z=X[actual]
        err=e[actual].astype(bool)
        for j,k in enumerate(FEATURES):
            groupsout.append({"writer":w,"true_class":t,"feature":k,
                 "n":len(ix),"errors":int(err.sum()),"correct":int((~err).sum()),
                 "mean_wrong":float(z[err,j].mean()) if err.any() else "",
                 "mean_correct":float(z[~err,j].mean()) if (~err).any() else "",
                 "mean_wrong_minus_correct":float(z[err,j].mean()-z[~err,j].mean())
                    if err.any() and (~err).any() else ""})
    write_csv(out/"feature_comparison.csv",groupsout)
    return rows


def select_focus(feats):
    selections=[]
    d=defaultdict(list)
    for r in feats:
        if r["true_label"] in FOCUS:
            d[(r["writer"],r["true_label"],r["combined_correct"])].append(r)
    for w in WRITERS:
        for cls in FOCUS:
            for corr in ("0","1"):
                arr=d[(w,cls,corr)]
                if arr:
                    # evenly distributed image indices; no model-score-based cherry picking
                    at=np.linspace(0,len(arr)-1,min(3,len(arr)),dtype=int)
                    selections.extend(arr[int(i)] for i in sorted(set(at)))
    keys={(r["writer"],r["index_in_writer"]) for r in selections}
    assert len(keys)==len(selections)
    return selections


def do_ig(pipe, x, cls, rival, steps=128):
    net=pipe.named_steps["model"]
    v=np.asarray(x,dtype=np.float64).reshape(784)/255.0
    w0,w1=(np.asarray(v,dtype=np.float64) for v in net.coefs_)
    b0,b1=(np.asarray(v,dtype=np.float64) for v in net.intercepts_)
    vclass=LABELS.index(cls); vrival=LABELS.index(rival)
    dw=w1[:,vclass]-w1[:,vrival]
    alpha=(np.arange(steps,dtype=np.float64)+.5)/steps
    h=alpha[:,None]*v[None,:]@w0 +b0
    grad=((h>0)*dw[None,:])@w0.T
    ig=v*grad.mean(axis=0)
    def margin(inp):
        z=np.maximum(inp@w0+b0,0.0)@w1+b1
        return float(z[vclass]-z[vrival])
    observed=margin(v);baseline=margin(np.zeros_like(v))
    residual=float(ig.sum()-(observed-baseline))
    return ig.reshape(28,28),observed,residual


def region_interventions(pipe, image, cls, competitor, seed=42):
    """Predeclared spatial regions vs same-count foreground removal (random controls).

    Region erasure may affect geometry; this tests coordinate sensitivity, NOT
    independent semantic stroke/connected-component structure.
    """
    xx=np.asarray(image,dtype=np.float32)
    mask=xx>=32
    yy,xxcoords=np.indices((28,28))
    regions={
        "left_middle":(xxcoords<=10)&(yy>=7)&(yy<=21),
        "right_middle":(xxcoords>=17)&(yy>=7)&(yy<=21),
        "center_vertical":(xxcoords>=11)&(xxcoords<=16)&(yy>=5)&(yy<=23),
        "center_horizontal":(yy>=10)&(yy<=17)&(xxcoords>=5)&(xxcoords<=22)
    }
    full=np.flatnonzero(mask)
    rng=np.random.default_rng(seed)
    base=logits(pipe, image)[0]
    ci,ri=LABELS.index(cls),LABELS.index(competitor)
    initial_margin=float(base[ci]-base[ri])
    rows=[]
    for name,region in regions.items():
        pos=np.flatnonzero(mask&region)
        if len(pos)<4 or len(full)-len(pos)<4:
            continue
        target=np.array(image,copy=True).reshape(784)
        target[pos]=0
        after=logits(pipe,target)[0]
        effect=float(after[ci]-after[ri]-initial_margin)
        rand=[]
        for _ in range(32):
            p=rng.choice(full,len(pos),replace=False)
            q=np.array(image,copy=True).reshape(784)
            q[p]=0
            margin=logits(pipe,q)[0]
            rand.append(float(margin[ci]-margin[ri]-initial_margin))
        rows.append({"region":name,"foreground_removed":len(pos),
                     "region_ink_mass_removed":float(np.asarray(image).reshape(784)[pos].sum()),
                     "region_delta_margin":effect,
                     "random_mean_delta_margin":float(np.mean(rand)),
                     "random_mean_abs_delta_margin":float(np.mean(np.abs(rand))),
                     "region_excess_abs_effect":float(abs(effect)-np.mean(np.abs(rand))),
                     "region_prediction_after":LABELS[int(after.argmax())],
                     "random_repetitions":len(rand)})
    return rows


def make_grids(feats, raw, out):
    figdir=out/"figures"
    figdir.mkdir(parents=True,exist_ok=True)
    for classes,name in ((("0","D"),"error_grid_0D"),(("8","B"),"error_grid_8B")):
        pool=[r for r in feats if r["true_label"] in classes]
        examples=[]
        for cls in classes:
            for incorrect in ("0","1"):
                sub=[r for r in pool if r["true_label"]==cls and r["combined_correct"]==incorrect]
                for w in WRITERS:
                    wr=[r for r in sub if r["writer"]==w]
                    if wr:examples.extend(wr[:2])
        examples=examples[:24]
        fig,axes=plt.subplots(4,6,figsize=(12,8))
        for ax,r in zip(axes.flat,examples):
            ax.imshow(raw[r["writer"]][r["index_in_writer"]],vmin=0,vmax=255,cmap="gray")
            ax.set_title(str(r["writer"])+"/"+str(r["index_in_writer"])+
                         ": "+r["true_label"]+" > "+r["combined_pred"],fontsize=8)
            ax.axis("off")
        for ax in axes.flat[len(examples):]:ax.axis("off")
        fig.suptitle("Real Hand-made | writer/index : actual > Combined prediction")
        fig.tight_layout()
        fig.savefig(figdir/(name+".png"),dpi=180)
        plt.close(fig)


def analysis(models, table, raw, feats, out):
    selections=select_focus(feats)
    igrows, regionrows=[],[]
    for r in selections:
        w,i,cls=r["writer"],r["index_in_writer"],r["true_label"]
        pipe=models["combined_heldout_"+str(w)+"_mlp.joblib"]
        normal=models["normal_mlp.joblib"]
        image=raw[w][i]
        rival=RIVAL[cls]
        for model_name,model in (("Normal",normal),("Combined",pipe)):
            ig,margin,residual=do_ig(model,image,cls,rival)
            score=logits(model,image)[0]
            igrows.append({"writer":w,"index_in_writer":i,"actual":cls,
                "rival":rival,"model":model_name,"correct":int(score.argmax()==LABELS.index(cls)),
                "prediction":LABELS[score.argmax()],"actual_minus_rival_logit_margin":margin,
                "IG_positive_sum":float(np.clip(ig,0,None).sum()),
                "IG_negative_sum":float(np.clip(ig,None,0).sum()),
                "IG_absolute_sum":float(np.abs(ig).sum()),
                "IG_completeness_residual":residual,
                "IG_max_coord_y":int(np.unravel_index(np.abs(ig).argmax(),ig.shape)[0]),
                "IG_max_coord_x":int(np.unravel_index(np.abs(ig).argmax(),ig.shape)[1])})
        for intervention in region_interventions(pipe,image,cls,rival,
                                                 seed=20261116+w*1000+i):
            regionrows.append({"writer":w,"index_in_writer":i,"actual":cls,
                "rival":rival,"baseline_prediction":r["combined_pred"],
                "baseline_correct":int(r["combined_correct"]),**intervention})
    write_csv(out/"margin_ig_diagnostics.csv",igrows)
    write_csv(out/"region_ablation_controls.csv",regionrows)
    return selections,igrows,regionrows


def summarize(manifest,pred,assoc,selections,igrows,regrows,wr,out,elapsed):
    corr = Counter()
    for r in pred:
        nc=r["normal_correct"]=="1";cc=r["combined_correct"]=="1"
        trans="both_correct" if nc and cc else "corrected" if not nc and cc else \
              "new_error" if nc and not cc else "both_wrong"
        corr[trans]+=1
    pairs={}
    for t,p in (("0","D"),("D","0"),("8","B"),("B","8")):
        rr=[r for r in pred if r["true_label"]==t and r["combined_pred"]==p]
        pairs[t+"->"+p]={"combined":len(rr),
                         "writer_breakdown":{str(w):sum(r["writer"]==w for r in rr)
                                             for w in WRITERS},
                         "new_error":sum(r["normal_pred"]==t for r in rr),
                         "same_wrong":sum(r["normal_pred"]==p for r in rr),
                         "other_wrong":sum(r["normal_pred"]!=t and r["normal_pred"]!=p
                                           for r in rr)}
    # A region ablation is an exploratory coordinate sensitivity signal only.
    region_group={}
    for region in sorted({r["region"] for r in regrows}):
        rr=[r for r in regrows if r["region"]==region]
        perimg=defaultdict(list)
        for r in rr:perimg[(r["writer"],r["index_in_writer"])].append(r["region_excess_abs_effect"])
        samplemeans=np.array([np.mean(v) for v in perimg.values()])
        if len(samplemeans)<2:continue
        rng=np.random.default_rng(20261116)
        boots=np.mean(rng.choice(samplemeans,(2000,len(samplemeans)),replace=True),axis=1)
        region_group[region]={"n_distinct_images":len(samplemeans),
            "mean_excess_abs_logit":float(samplemeans.mean()),
            "bootstrap95":np.quantile(boots,[.025,.975]).tolist(),
            "per_image_fraction_excess_positive":float(np.mean(samplemeans>0))}
    report={
        "status":"experimental_evidence_created; A/B go/no-go requires interpretation",
        "source":"EXP-CMB-001 original OOF and frozen models",
        "n_images":2560, "n_focus_images":640,
        "baseline_oof_accuracy":2080/2560,
        "transitions":dict(corr),"focus_pairs":pairs,
        "feature_associations":assoc,
        "A_IG_n_model_evaluations":len(igrows),
        "A_distinct_images":len(selections),
        "A_max_abs_IG_completeness_residual":max(abs(x["IG_completeness_residual"])
                                                for x in igrows),
        "A_region_ablation_exploratory":region_group,
        "writer_metrics":wr,
        "rules":{"B":"Look for within-writer-and-class BH q<0.05, nontrivial effect, across-writer consistency, and threshold robustness before recommending spatial normalization",
                 "A":"Logit/IG and prespecified region ablation show input coordinate relevance only. To claim meaningful stroke topology requires structure-preserving controls. Do not equate signed IG to semantic shape.",
                 "decision":"A/B are not mutually exclusive. If evidence cannot distinguish pathways, report INCONCLUSIVE and extend EXP-CMB-002, not advance."},
        "environment":{"numpy":np.__version__,"sklearn":sklearn.__version__,"scipy":scipy.__version__},
        "elapsed_seconds":elapsed,
        "original_test_used":False,"professor_test_used":False,
        "new_training_runs":0,"model_sha_members":manifest["models_zip_members_sha256"]
    }
    (out/"evidence_summary.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    summary=[
        "# EXP-CMB-002 evidence — automatic execution results (research review)",
        "",
        "**Status:** analysis completed, final A/B determination pending interpretation and controls.",
        "",
        "- Original OOF revalidated: 2,560 Hand-made images, Combined 2,080 correct (81.25%).",
        "- Corrected "+str(corr["corrected"])+", newly wrong "+str(corr["new_error"])+
          ", both wrong "+str(corr["both_wrong"])+".",
        "- Focused true classes 0/D/8/B: 640 images.",
        "- Fixed-fold model IG diagnostics: "+str(len(selections))+" distinct images, "+
          str(len(igrows))+" model-condition rows.",
        "- Stratified writer+class permutation: 2,500 repetitions per global feature, BH-FDR across 7 features.",
        "- Region ablation: predefined regions and 32 random same-foreground-count removals per region.",
        "- **Interpretation limits:** region ablation modifies geometry and brightness mass; it is NOT a pure stroke-shape intervention. B correlation is NOT proof of causal improvement. No new training.",
        "",
        "## Feature association (pooled 0/D/8/B, conditioned on writer and actual class)",
        "",
        "| Feature | error - correct (within-group z) | permutation p | BH q |",
        "|---|---:|---:|---:|",
    ]
    summary.extend("| {} | {:.3f} | {:.4f} | {:.4f} |".format(x["feature"],
         x["error_minus_correct_mean_z"],x["stratified_permutation_p"],x["bh_fdr_q"])
         for x in assoc)
    summary.extend(["","## Region ablation (exploratory coordinate relevance only)","",
          "| Region | Images | Mean excess abs margin effect | image bootstrap 95% interval |",
          "|---|---:|---:|---|"])
    summary.extend("| {} | {} | {:.3f} | [{:.3f}, {:.3f}] |".format(k,v["n_distinct_images"],
                     v["mean_excess_abs_logit"],*v["bootstrap95"])
                   for k,v in region_group.items())
    summary.extend(["","See evidence_summary.json, feature_comparison.csv, margin_ig_diagnostics.csv, region_ablation_controls.csv and actual real-image grids.",
      "","Official original Test and professor Test were not accessed. Baseline EXP-CMB-001 files were not changed."])
    (out/"SUMMARY.md").write_text("\n".join(summary)+"\n",encoding="utf-8")


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()
    out=args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    start=time.perf_counter()
    manifest,pred,table,raw,labels=load_inputs()
    models=load_models(manifest)
    for w in WRITERS:
        X=raw[w].reshape(640,784).astype(np.float32)
        for key,model in (("normal_pred",models["normal_mlp.joblib"]),
                          ("combined_pred",models["combined_heldout_"+str(w)+"_mlp.joblib"])):
            obtained=np.array(list(LABELS))[model.predict(X).astype(int)]
            expected=np.array([table[w,i][key] for i in range(640)])
            assert np.array_equal(obtained,expected),("stored OOF prediction mismatch",w,key)
    print("REPRODUCED 2,560 OOF predictions with original frozen 5 models",flush=True)
    feats=prepare_features(pred,raw)
    write_csv(out/"image_features.csv",feats)
    c,writer_cms,wrows,trows=make_class_tables(pred,out)
    save_cms(c,writer_cms,out)
    assoc=stratified_permutation(feats,out)
    make_grids(feats,raw,out)
    selections,igrows,regrows=analysis(models,table,raw,feats,out)
    summarize(manifest,pred,assoc,selections,igrows,regrows,wrows,out,
              time.perf_counter()-start)
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)
    print("EXPERIMENT COMPLETED — FROZEN MODELS, NO TEST",flush=True)


if __name__=="__main__":
    main()


## exp_cmb002_followup

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb002_followup.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb002_followup.py)  
Git blob: `a71354ef25382cc2f7a7d16955931064a996a4ee`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-002 continuation: writer-specific morphology + matched pixel swaps.

Purpose: decide whether further investigation of (A) pixel/stroke structure and
(B) global geometry/ink is justified. Frozen Combined 4-fold models only.
No Combined training, no Original Test, no training-time parameter selection.
Results are exploratory and remain on a review branch before user approval.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb002_followup.py
#!/usr/bin/env python3
"""EXP-CMB-002 continuation: writer-specific morphology + matched pixel swaps.

Purpose: decide whether further investigation of (A) pixel/stroke structure and
(B) global geometry/ink is justified. Frozen Combined 4-fold models only.
No Combined training, no Original Test, no training-time parameter selection.
Results are exploratory and remain on a review branch before user approval.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path
import numpy as np
from scipy import ndimage
import exp_cmb002_evidence as core

FOCUS=core.FOCUS
RIVAL=core.RIVAL
FEATURES_MORPH=("holes_128","connected_components_128","left_mass_fraction",
  "right_mass_fraction","middle_mass_fraction","upper_mass_fraction",
  "central_row_mass_fraction","left_column_mass_fraction",
  "bilateral_asymmetry","upper_lower_asymmetry",
  "centerline_foreground_128")
TARGETS=((43,"0"),(41,"D"),(42,"D"),(43,"8"),(41,"B"))
ALL_GROUPS=tuple((w,c) for w in core.WRITERS for c in FOCUS)


def morphological(img):
    a=np.asarray(img,dtype=np.float64)
    mass=max(float(a.sum()),1e-12)
    mask=a>=128
    struct=ndimage.generate_binary_structure(2,1)
    components,ncomp=ndimage.label(mask,structure=struct)
    bg,nlab=ndimage.label(~mask,structure=struct)
    exterior=set(np.unique(np.r_[bg[0,:],bg[-1,:],bg[:,0],bg[:,-1]]).tolist())
    holes=sum(i not in exterior for i in range(1,nlab+1))
    return {
        "holes_128":holes,
        "connected_components_128":ncomp,
        "left_mass_fraction":float(a[:,:11].sum()/mass),
        "right_mass_fraction":float(a[:,17:].sum()/mass),
        "middle_mass_fraction":float(a[:,11:17].sum()/mass),
        "upper_mass_fraction":float(a[:14,:].sum()/mass),
        "central_row_mass_fraction":float(a[10:18,:].sum()/mass),
        "left_column_mass_fraction":float(a[6:23,7:13].sum()/mass),
        "bilateral_asymmetry":float(np.abs(a-a[:,::-1]).sum()/mass),
        "upper_lower_asymmetry":float(abs(a[:14,:].sum()-a[14:,:].sum())/mass),
        "centerline_foreground_128":int(mask[8:21,12:16].sum())
    }


def permutation_table(rows,features,groups,repeats,seed):
    """Within true class and writer, compare error and correct via shuffles."""
    rng=np.random.default_rng(seed)
    n=len(rows)
    x=np.array([[float(row[k]) for k in features] for row in rows],dtype=float)
    y=np.array([int(row["combined_correct"]=="0") for row in rows])
    by=defaultdict(list)
    for i,r in enumerate(rows):by[(r["writer"],r["true_label"])].append(i)
    assert set(by)==set(groups)
    # Z-score within writer/class to avoid writer/class mean confounding.
    z=np.zeros_like(x)
    ycenter=np.zeros(n)
    for ids in by.values():
        u=x[ids]
        z[ids]=(u-u.mean(0))/np.maximum(u.std(0),1e-6)
        ycenter[ids]=y[ids]-y[ids].mean()
    obs=z.T@ycenter/n
    sim=np.zeros((repeats,len(features)))
    for b in range(repeats):
        yy=np.zeros(n)
        for ids in by.values():
            yy[ids]=rng.permutation(y[ids])-y[ids].mean()
        sim[b]=z.T@yy/n
    p=(1+(np.abs(sim)>=np.abs(obs)).sum(0))/(repeats+1)
    order=np.argsort(p)
    q=np.ones(len(p))
    best=1.0
    for irank in range(len(order)-1,-1,-1):
        idx=order[irank]
        best=min(best,float(p[idx])*len(p)/(irank+1))
        q[idx]=best
    return [{"feature":ft,"n":n,"errors":int(y.sum()),
             "error_minus_correct_z":float(z[y==1,j].mean()-z[y==0,j].mean()),
             "p":float(p[j]),"q":float(q[j]),"strata":len(by)}
            for j,ft in enumerate(features)]


def targeted_permutations(rows,features,repeats=2000):
    raw=[]
    for w,cls in TARGETS:
        rr=[r for r in rows if r["writer"]==w and r["true_label"]==cls]
        assert len(rr)==40
        err=np.array([int(r["combined_correct"]=="0") for r in rr],dtype=float)
        assert 0<err.sum()<40
        x=np.array([[float(r[ft]) for ft in features] for r in rr])
        rng=np.random.default_rng(20261116+100*w+core.LABELS.index(cls))
        std=np.maximum(x.std(0),1e-6)
        z=(x-x.mean(0))/std
        obs=(z.T@(err-err.mean()))/40
        sims=np.zeros((repeats,len(features)))
        for i in range(repeats):
            sims[i]=(z.T@(rng.permutation(err)-err.mean()))/40
        p=(1+(np.abs(sims)>=np.abs(obs)).sum(0))/(repeats+1)
        for j,ft in enumerate(features):
            raw.append({"writer":w,"true_class":cls,"feature":ft,
                       "n":40,"n_wrong":int(err.sum()),"n_correct":int(40-err.sum()),
                       "wrong_minus_correct_raw":float(x[err==1,j].mean()-x[err==0,j].mean()),
                       "effect_wrong_minus_correct_z":float(z[err==1,j].mean()-z[err==0,j].mean()),
                       "p_unadjusted":float(p[j])})
    order=sorted(range(len(raw)),key=lambda i:raw[i]["p_unadjusted"])
    best=1.0
    for irank in range(len(order)-1,-1,-1):
        idx=order[irank]
        best=min(best,raw[idx]["p_unadjusted"]*len(raw)/(irank+1))
        raw[idx]["q_5_groups_all_features"]=best
    return raw


def n_edges(mask):
    a=np.asarray(mask,dtype=bool)
    return int(np.sum(a[:,:-1]&a[:,1:])+np.sum(a[:-1,:]&a[1:,:]))


def swap_candidates(image,seed,target=54):
    """Fixed 4 foreground/background exchanges per candidate; exact intensity histogram.

    Swaps are entirely within the original bbox and near neighbours. Accept only
    candidates whose bbox (threshold 128) remains exactly and CoM shift <=0.3px.
    Candidate selection uses no MLP outputs or ground-truth label.
    """
    x=np.asarray(image,dtype=np.uint8)
    m=x>=128
    ys,xs=np.nonzero(m)
    if len(xs)<10:return []
    xmin,xmax=int(xs.min()),int(xs.max())
    ymin,ymax=int(ys.min()),int(ys.max())
    bright=np.flatnonzero((x>=200)&np.indices(x.shape)[0].__ge__(ymin+1)&
        (np.indices(x.shape)[0]<=ymax-1)&
        (np.indices(x.shape)[1]>=xmin+1)&
        (np.indices(x.shape)[1]<=xmax-1))
    dark=np.flatnonzero((x<=30)&(np.indices(x.shape)[0]>=ymin+1)&
        (np.indices(x.shape)[0]<=ymax-1)&
        (np.indices(x.shape)[1]>=xmin+1)&
        (np.indices(x.shape)[1]<=xmax-1))
    if len(bright)<4 or len(dark)<4:return []
    yy,xx=np.unravel_index(bright,(28,28))
    yd,xd=np.unravel_index(dark,(28,28))
    pairs=[]
    for i,fg in enumerate(bright):
        dist=np.abs(xd-xx[i])+np.abs(yd-yy[i])
        for j in np.flatnonzero((dist>=1)&(dist<=3)):
            pairs.append((int(fg),int(dark[j]),int(dist[j])))
    if len(pairs)<16:return []
    rng=np.random.default_rng(seed)
    original_edges=n_edges(m)
    a=x.ravel()
    total=float(x.sum())
    if total<1:return []
    init_cx=float((x.astype(float)*np.arange(28)[None,:]).sum()/total)
    init_cy=float((x.astype(float)*np.arange(28)[:,None]).sum()/total)
    saved={}
    for attempt in range(2400):
        picks=rng.permutation(len(pairs))
        chosen=[]
        used=set()
        for pos in picks[:min(len(picks),160)]:
            fg,bg,d=pairs[pos]
            if fg not in used and bg not in used:
                chosen.append((fg,bg,d));used.add(fg);used.add(bg)
                if len(chosen)==4:break
        if len(chosen)<4:continue
        b=a.copy()
        for fg,bg,_ in chosen:b[fg],b[bg]=b[bg],b[fg]
        shifted=b.reshape(28,28)
        ys2,xs2=np.nonzero(shifted>=128)
        if not len(xs2) or (int(xs2.min()),int(xs2.max()),int(ys2.min()),int(ys2.max()))!=(xmin,xmax,ymin,ymax):continue
        new_total=float(b.sum())
        assert new_total==total
        cx=float((shifted.astype(float)*np.arange(28)[None,:]).sum()/total)
        cy=float((shifted.astype(float)*np.arange(28)[:,None]).sum()/total)
        shift=float(np.hypot(cx-init_cx,cy-init_cy))
        if shift>0.3:continue
        # Preserve complete intensity histogram and foreground count.
        assert np.array_equal(np.sort(a),np.sort(b))
        assert int((shifted>=128).sum())==int(m.sum())
        key=tuple(sorted((fg,bg) for fg,bg,_ in chosen))
        if key in saved:continue
        saved[key]={
            "array":shifted.copy(),
            "delta_4neighbour_connections":n_edges(shifted>=128)-original_edges,
            "mean_manhattan_swap_distance":float(np.mean([d for _,_,d in chosen])),
            "center_shift_px":shift
        }
        if len(saved)>=target:break
    return list(saved.values())


def select_samples(rows):
    d=defaultdict(list)
    for r in rows:
        if r["true_label"] in FOCUS:
            d[(r["writer"],r["true_label"],r["combined_correct"])].append(r)
    sel=[]
    for w,c in ALL_GROUPS:
        for correct in ("0","1"):
            v=d[(w,c,correct)]
            if v:
                at=np.linspace(0,len(v)-1,min(2,len(v)),dtype=int)
                sel.extend(v[int(j)] for j in sorted(set(at)))
    assert len({(r["writer"],r["index_in_writer"]) for r in sel})==len(sel)
    return sel


def margin(model,image,actual,rival):
    logits=core.logits(model,image)[0]
    return float(logits[core.LABELS.index(actual)]-logits[core.LABELS.index(rival)]),core.LABELS[int(logits.argmax())]


def topology_analysis(rows,raw,models,out):
    selected=select_samples(rows)
    trials,paired=[],[]
    for row in selected:
        w,idx,cls=row["writer"],row["index_in_writer"],row["true_label"]
        mod=models["combined_heldout_"+str(w)+"_mlp.joblib"]
        im=raw[w][idx]
        baseline,p0=margin(mod,im,cls,RIVAL[cls])
        proposals=swap_candidates(im,20261116+w*200+idx,target=54)
        if len(proposals)<16:continue
        # Candidate rank based only on adjacent-foreground graph, no MLP scores.
        ds=np.array([p["delta_4neighbour_connections"] for p in proposals])
        distances=np.array([p["mean_manhattan_swap_distance"] for p in proposals])
        order=np.argsort(ds,kind="stable")
        nselect=8
        disruptive=order[:nselect]
        controls=[]
        available=set(range(len(proposals)))-set(disruptive)
        for i in disruptive:
            # Match overall displacement (same 4 swaps, similar mean Manhattan length).
            j=min(available,key=lambda j:(abs(distances[j]-distances[i]),abs(ds[j]-np.median(ds)),j))
            controls.append(j)
            available.remove(j)
        this=[]
        for condition,indices in (("largest_connectivity_loss",disruptive),
                                  ("distance_matched_control",controls)):
            for j in indices:
                p=proposals[int(j)]
                after,pred=margin(mod,p["array"],cls,RIVAL[cls])
                trial={"writer":w,"index_in_writer":idx,"actual":cls,
                       "baseline_correct":int(row["combined_correct"]),
                       "condition":condition,"candidate_id":int(j),
                       "baseline_margin":baseline,"margin_delta":after-baseline,
                       "abs_margin_delta":abs(after-baseline),
                       "new_prediction":pred,"prediction_flip":int(pred!=p0),
                       "connectivity_change":p["delta_4neighbour_connections"],
                       "mean_swap_distance":p["mean_manhattan_swap_distance"],
                       "center_shift_px":p["center_shift_px"],
                       "foreground_swaps":4,"pixels_changed":8}
                trials.append(trial);this.append(trial)
        a=[x for x in this if x["condition"]=="largest_connectivity_loss"]
        b=[x for x in this if x["condition"]=="distance_matched_control"]
        paired.append({"writer":w,"index_in_writer":idx,"actual":cls,
          "baseline_correct":int(row["combined_correct"]),
          "n_valid_proposals":len(proposals),
          "mean_edges_loss_selected":float(np.mean([x["connectivity_change"] for x in a])),
          "mean_edges_loss_control":float(np.mean([x["connectivity_change"] for x in b])),
          "mean_distance_selected":float(np.mean([x["mean_swap_distance"] for x in a])),
          "mean_distance_control":float(np.mean([x["mean_swap_distance"] for x in b])),
          "mean_abs_delta_selected":float(np.mean([x["abs_margin_delta"] for x in a])),
          "mean_abs_delta_control":float(np.mean([x["abs_margin_delta"] for x in b])),
          "delta_abs_margin_selected_minus_control":float(np.mean([x["abs_margin_delta"] for x in a])-
                                                       np.mean([x["abs_margin_delta"] for x in b])),
          "flip_selected":float(np.mean([x["prediction_flip"] for x in a])),
          "flip_control":float(np.mean([x["prediction_flip"] for x in b]))})
    if not paired:raise RuntimeError("No valid strict histogram-preserving proposals")
    core.write_csv(out/"pixel_swap_trials.csv",trials)
    core.write_csv(out/"pixel_swap_per_image.csv",paired)
    summaries=[]
    for label,sel in (
        ("all",paired),
        ("0D",[r for r in paired if r["actual"] in ("0","D")]),
        ("8B",[r for r in paired if r["actual"] in ("8","B")])):
        if len(sel)<5:continue
        d=np.array([r["delta_abs_margin_selected_minus_control"] for r in sel])
        rng=np.random.default_rng(20261116)
        boot=np.mean(rng.choice(d,size=(4000,len(d)),replace=True),axis=1)
        summaries.append({"group":label,"n_independent_original_images":len(sel),
          "mean_excess_abs_margin":float(d.mean()),
          "bootstrap95_low":float(np.quantile(boot,.025)),
          "bootstrap95_high":float(np.quantile(boot,.975)),
          "fraction_per_image_positive":float(np.mean(d>0)),
          "mean_edges_selected":float(np.mean([r["mean_edges_loss_selected"] for r in sel])),
          "mean_edges_control":float(np.mean([r["mean_edges_loss_control"] for r in sel])),
          "mean_move_selected":float(np.mean([r["mean_distance_selected"] for r in sel])),
          "mean_move_control":float(np.mean([r["mean_distance_control"] for r in sel])),
          "candidate_count_min":min(r["n_valid_proposals"] for r in sel),
          "candidate_count_max":max(r["n_valid_proposals"] for r in sel),
          "n_each_condition":8})
    core.write_csv(out/"topology_contrasts.csv",summaries)
    return len(selected),len(paired),summaries



def pair_specific_checks(rows, repeats=4000):
    """Exploratory direct-error vs correct controls, sensitivity to threshold.

    This is POST-HOC after viewing the baseline and initial follow-up, not an
    independent confirmatory test; interpret adjusted p-values accordingly.
    """
    specs=(
        (43,"8","B",("width","width_t64","width_t128",
                     "ink_ratio","ink_ratio_t64","ink_ratio_t128",
                     "middle_mass_fraction")),
        (41,"D","0",("cx","cx_t64","cx_t128","left_mass_fraction","width")),
        (43,"0","D",("cy","cy_t64","cy_t128","width","ink_ratio")),
        (41,"B","8",("width","ink_ratio","middle_mass_fraction"))
    )
    results=[]
    for w,actual,wrong,variables in specs:
        group=[r for r in rows if r["writer"]==w and r["true_label"]==actual
               and r["combined_pred"] in (actual,wrong)]
        y=np.asarray([r["combined_pred"]==wrong for r in group],dtype=bool)
        assert len(group)>=20 and y.sum()>=4 and (~y).sum()>=4
        arr=np.asarray([[float(r[ft]) for ft in variables] for r in group])
        rng=np.random.default_rng(20261116+1000*w+core.LABELS.index(actual))
        obs=arr[y].mean(axis=0)-arr[~y].mean(axis=0)
        pcount=np.zeros(len(variables),dtype=int)
        for trial in range(repeats):
            perm=rng.permutation(y)
            diff=arr[perm].mean(axis=0)-arr[~perm].mean(axis=0)
            pcount+=(np.abs(diff)>=np.abs(obs))
        p=(pcount+1)/(repeats+1)
        for j,ft in enumerate(variables):
            results.append({"writer":w,"actual":actual,"wrong_prediction":wrong,
             "feature":ft,"n_wrong":int(y.sum()),"n_correct":int((~y).sum()),
             "wrong_mean":float(arr[y,j].mean()),"correct_mean":float(arr[~y,j].mean()),
             "wrong_minus_correct":float(obs[j]),"perm_p_posthoc":float(p[j]),
             "permutation_count":repeats})
    idx=sorted(range(len(results)),key=lambda i:results[i]["perm_p_posthoc"])
    q=1.0
    for rank in range(len(idx)-1,-1,-1):
        i=idx[rank]
        q=min(q,results[i]["perm_p_posthoc"]*len(idx)/(rank+1))
        results[i]["BH_q_across_posthoc_tests"]=float(q)
    return results


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--out",type=Path,required=True)
    args=parser.parse_args()
    out=args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    manifest,preds,table,raw,labels=core.load_inputs()
    models=core.load_models(manifest)
    rows=core.prepare_features(preds,raw)
    for r in rows:
        r.update(morphological(raw[r["writer"]][r["index_in_writer"]]))
    core.write_csv(out/"morphology_features.csv",(
        {k:r[k] for k in ("writer","index_in_writer","true_label","combined_correct",
                          *FEATURES_MORPH)} for r in rows))
    focus=[r for r in rows if r["true_label"] in FOCUS]
    assert len(focus)==640
    morph=permutation_table(focus,FEATURES_MORPH,ALL_GROUPS,3000,20261116)
    core.write_csv(out/"morphology_associations.csv",morph)
    target=targeted_permutations(focus,(*core.FEATURES,*FEATURES_MORPH),repeats=2000)
    core.write_csv(out/"targeted_writer_class_tests.csv",target)
    selected,nvalid,topology=topology_analysis(focus,raw,models,out)
    pair_checks=pair_specific_checks(focus)
    core.write_csv(out/"pair_specific_threshold_sensitivity.csv",pair_checks)
    summary={"status":"followup completed; interpret with explicit A/B decision",
       "n_focus_images":640,"n_morph_features":len(FEATURES_MORPH),
       "morphology_associations":morph,
       "targeted_unadjusted_q_significant":[x for x in target if x["q_5_groups_all_features"]<.05],
       "targeted_five_groups":[list(v) for v in TARGETS],
       "topology_selected_images":selected,"topology_valid_images":nvalid,
       "topology_contrasts":topology,
       "posthoc_pair_specific_threshold_tests":pair_checks,
       "caveats":[
        "OOF repeatedly investigated as development data; not independent validation.",
        "Morphology descriptors overlap global geometry and do not isolate human semantic strokes.",
        "Pixel swaps preserve full intensity histogram, bbox and >=128 foreground count, with pixel-shift CoM <=0.3, but exact continuous CoM and all local geometry cannot be held fixed.",
        "An adjacency-edge count is a topology proxy, not a human visual recognition score.",
        "Targeted five strata were chosen after observing original OOF failure patterns; report them as exploratory and control FDR.",
        "The models were never retrained or tuned."
       ],
       "official_original_test_used":False,"professor_test_used":False}
    (out/"FOLLOWUP_EVIDENCE.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    lines=["# EXP-CMB-002 continuation — morphology and controlled pixel swaps","",
      "- Original held-out OOF and 5 frozen models re-read; no MLP training.",
      "- Same 640 focus images. Global morphology feature permutations: 3,000 within-writer-and-class shuffles, BH across 11.",
      "- Five preselected writer/class strata: 2,000 shuffles, BH across all 5×18 tests.",
      "- Pixel swap samples: "+str(selected)+" selected, "+str(nvalid)+" with strict valid proposals, full 0–255 histogram and bbox preserved.",
      "",
      "## Global morphology association","",
      "| Feature | error-minus-correct z | p | q |","|---|---:|---:|---:|"]
    lines.extend("| {} | {:.3f} | {:.4f} | {:.4f} |".format(r["feature"],
         r["error_minus_correct_z"],r["p"],r["q"]) for r in morph)
    lines.extend(["","## Image-level topology intervention contrasts","",
      "| Group | N images | Selected - control absolute margin | Bootstrap 95% CI | Mean connection change (selected / control) |",
      "|---|---:|---:|---|---|"])
    lines.extend("| {} | {} | {:.3f} | [{:.3f}, {:.3f}] | {:.2f} / {:.2f} |".format(
        r["group"],r["n_independent_original_images"],r["mean_excess_abs_margin"],
        r["bootstrap95_low"],r["bootstrap95_high"],r["mean_edges_selected"],
        r["mean_edges_control"]) for r in topology)
    sig=[x for x in target if x["q_5_groups_all_features"]<.05]
    lines.extend(["","## Targeted writer-class tests","",
       "BH q<0.05 among 5 selected strata × 18 features: "+str(len(sig))+".",
       "This is **exploratory** even if significant; selected strata arise from baseline errors.",
       "See targeted_writer_class_tests.csv for all adjusted p-values.",
       "Direct error-pair versus correct-control threshold checks are in pair_specific_threshold_sensitivity.csv (post-hoc; BH across all checks).",
       "",
       "**Limits:** the pixel swaps are matched on 4 exchanges and displacement and preserve histogram, bounding box, thresholded foreground pixel count, with <=0.3px CoM movement. Four-neighbour connectivity is only a morphological proxy. No final causal claim of semantic shape recognition.",
       ""])
    (out/"FOLLOWUP_SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    print((out/"FOLLOWUP_SUMMARY.md").read_text(encoding="utf-8"),flush=True)
    print("EXP-CMB-002 FOLLOWUP COMPLETE — NO TRAINING OR TEST",flush=True)


if __name__=="__main__":main()


## exp_cmb003_input_sensitivity

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb003_input_sensitivity.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb003_input_sensitivity.py)  
Git blob: `bed42ae58ea76784e58a997cad4d8e8252b670f6`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-003: 10 controlled input perturbations vs 5 frozen EXP-CMB-001 MLPs.

Scientific scope: diagnose whether the observed writer/class geometry & ink
differences cause meaningful corrections under controlled, label-blind changes.
No training. No Original Test, professor Test or new input-derived fitting.
The same transformation is applied to all images for each condition.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb003_input_sensitivity.py
#!/usr/bin/env python3
"""EXP-CMB-003: 10 controlled input perturbations vs 5 frozen EXP-CMB-001 MLPs.

Scientific scope: diagnose whether the observed writer/class geometry & ink
differences cause meaningful corrections under controlled, label-blind changes.
No training. No Original Test, professor Test or new input-derived fitting.
The same transformation is applied to all images for each condition.
"""
from __future__ import annotations
import argparse
import csv
import gzip
import hashlib
import json
import math
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from scipy.ndimage import maximum_filter, minimum_filter, affine_transform
from scipy.stats import binomtest
import scipy
import sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

ROOT=Path(__file__).resolve().parents[4]
COMBINED=ROOT/"05_model_optimize"/"MLP"/"Combined"
sys.path.insert(0,str(COMBINED/"scripts"))
import exp_cmb002_evidence as core

WRITERS=core.WRITERS
LABELS=core.LABELS
FOCUS=(("0","D"),("D","0"),("8","B"),("B","8"))
PRIMARY=(("41","D","0"),("43","8","B"),("43","0","D"),("41","B","8"))

@dataclass(frozen=True)
class Variant:
    name:str
    family:str
    value:object

VARIANTS=[
    Variant("original","original",None),
    Variant("shift_L1","shift",(-1,0)),
    Variant("shift_R1","shift",(1,0)),
    Variant("shift_U1","shift",(0,-1)),
    Variant("shift_D1","shift",(0,1)),
    Variant("scale_095","scale",.95),
    Variant("scale_105","scale",1.05),
    Variant("stroke_thin_25","stroke",("thin",.25)),
    Variant("stroke_thick_25","stroke",("thick",.25)),
    Variant("brightness_090","brightness",.90),
    Variant("brightness_110","brightness",1.10),
]
assert len(VARIANTS)==11

def transform(batch,condition):
    """Exact Normal Input_Sensitivity transform for matching variants."""
    a=np.asarray(batch,dtype=np.float32).reshape(-1,28,28)
    if condition.family=="original":
        z=a.copy()
    elif condition.family=="shift":
        dx,dy=condition.value
        z=np.zeros_like(a)
        ys=slice(max(0,-dy),min(28,28-dy))
        yd=slice(max(0,dy),min(28,28+dy))
        xs=slice(max(0,-dx),min(28,28-dx))
        xd=slice(max(0,dx),min(28,28+dx))
        z[:,yd,xd]=a[:,ys,xs]
    elif condition.family=="brightness":
        z=np.clip(a*float(condition.value),0,255)
    elif condition.family=="stroke":
        direction,fraction=condition.value
        if direction=="thick":
            modified=maximum_filter(a,size=(1,3,3),mode="constant",cval=0)
        else:
            modified=minimum_filter(a,size=(1,3,3),mode="constant",cval=0)
        z=(1-fraction)*a+fraction*modified
    elif condition.family=="scale":
        inv=1.0/float(condition.value)
        offset=13.5*(1.0-inv)
        z=affine_transform(a,matrix=np.diag([1,inv,inv]),
                           offset=(0.0,offset,offset),
                           output_shape=a.shape,order=1,mode="constant",
                           cval=0.0,prefilter=False)
    else:
        raise ValueError(condition.family)
    return np.rint(np.clip(z,0,255)).astype(np.float32).reshape(-1,784)

def self_test():
    x=np.zeros((2,784),dtype=np.float32)
    x[0,13*28+14]=255
    x[1,0]=255
    assert np.array_equal(transform(x,VARIANTS[0]),x)
    assert transform(x,VARIANTS[1])[1].sum()==0
    for v in VARIANTS:
        q=transform(x,v)
        assert q.shape==x.shape and q.dtype==np.float32
        assert q.min()>=0 and q.max()<=255
    print("TRANSFORM SELF-TEST 11 CONDITIONS PASSED",flush=True)

def load_official(manifest):
    path=ROOT/manifest["input_data_paths"]["official"]
    assert core.sha256(path)==manifest["dataset_sha256"]["official_v3"]
    split=COMBINED/"results"/"EXP_CMB_001"/"official_split_indices.npz"
    with np.load(split,allow_pickle=False) as d:
        train=d["train_idx"].astype(int)
        val=d["val_idx"].astype(int)
        train_group=d["train_group_ids"].astype(int)
        val_group=d["val_group_ids"].astype(int)
    assert (len(train),len(val))==(76800,19200)
    assert len(np.unique(np.r_[train,val]))==96000
    assert not np.intersect1d(train_group,val_group).size
    # Only read 96,000 official TRAIN arrays. NEVER access any *test* key.
    with np.load(path,allow_pickle=False) as d:
        assert "X_train" in d and "y_train" in d
        X=d["X_train"][val].reshape(-1,784).astype(np.float32)
        y=d["y_train"][val].astype(int)
    assert X.shape==(19200,784) and np.bincount(y,minlength=16).tolist()==manifest["official_split"]["validation_class_counts_0_to_15"]
    return X,y

def metrics(y,p,classes=None):
    scores=precision_recall_fscore_support(y,p,labels=(np.arange(16) if classes is None else classes),zero_division=0)
    prec,rec,f1,_=scores
    return {"n":len(y),"accuracy":float(np.mean(y==p)),
            "macro_precision":float(prec.mean()),"macro_recall":float(rec.mean()),
            "macro_f1":float(f1.mean()),"correct":int(np.sum(y==p))}

def paired_counts(y,before,after):
    was=before==y
    now=after==y
    c=int(np.sum(~was&now))
    i=int(np.sum(was&~now))
    f=int(np.sum(before!=after))
    wrongchanged=int(np.sum(~was&~now&(before!=after)))
    p=float(binomtest(c,c+i,p=.5).pvalue) if c+i else 1.0
    return {"corrections":c,"induced_errors":i,"net_correct":c-i,
            "prediction_flip_count":f,"prediction_flip_rate":f/len(y),
            "wrong_to_other_wrong":wrongchanged,"mcnemar_exact_p":p}

def bh(rows,key="mcnemar_exact_p"):
    idx=np.argsort([r[key] for r in rows])
    q=1.0
    for rank in range(len(rows)-1,-1,-1):
        i=idx[rank]
        q=min(q,rows[i][key]*len(rows)/(rank+1))
        rows[i]["BH_q_10_conditions"]=float(q)

def plot_cm(y,p,out,name):
    cm=confusion_matrix(y,p,labels=np.arange(16))
    for mode in ("count","normalized"):
        val=cm if mode=="count" else 100*cm/np.maximum(cm.sum(1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(9,8))
        maxcolor=160 if mode=="count" else 100
        im=ax.imshow(val,cmap="viridis",vmin=0,vmax=maxcolor)
        fig.colorbar(im,ax=ax,fraction=.045,pad=.03)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),xticklabels=list(LABELS),
               yticklabels=list(LABELS),xlabel="Predicted",ylabel="Actual",
               title=f"Combined OOF | {name} | {mode}")
        for i in range(16):
            for j in range(16):
                if cm[i,j]:
                    ax.text(j,i,("%.1f"%val[i,j] if mode=="normalized" else str(cm[i,j])),
                            ha="center",va="center",fontsize=5.5,
                            color="black" if val[i,j]>.53*maxcolor else "white")
        fig.tight_layout()
        fig.savefig(out/f"{name}_{mode}.png",dpi=160)
        plt.close(fig)

def plot_tradeoff(rows,out):
    v=[r for r in rows if r["variant"]!="original" and r["model"]=="Combined"]
    fig,ax=plt.subplots(figsize=(9,5))
    x=np.arange(len(v))
    ax.bar(x-.2,[r["corrections"] for r in v],width=.4,label="Wrong→correct")
    ax.bar(x+.2,[r["induced_errors"] for r in v],width=.4,label="Correct→wrong")
    ax.set(xticks=x,xticklabels=[r["variant"] for r in v],
           ylabel="OOF image count",title="EXP-CMB-003 | Corrections vs induced errors")
    plt.setp(ax.get_xticklabels(),rotation=45,ha="right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out/"corrections_vs_induced.png",dpi=180)
    plt.close(fig)


def plot_examples(raw,labels,models,figdir):
    """Representative *actual* held-out images, never used for condition selection."""
    examples=((43,32,"0"),(43,8,"8"),(41,13,"D"))
    choices=(VARIANTS[0],VARIANTS[5],VARIANTS[7],VARIANTS[1])
    fig,axes=plt.subplots(len(examples),len(choices),figsize=(10.8,8.3))
    for i,(writer,index,text) in enumerate(examples):
        assert LABELS[int(labels[writer][index])]==text
        original=raw[writer][index].reshape(1,784).astype(np.float32)
        model=models["combined_heldout_"+str(writer)+"_mlp.joblib"]
        for j,v in enumerate(choices):
            z=transform(original,v)
            pred=LABELS[int(model.predict(z)[0])]
            ax=axes[i,j]
            ax.imshow(z.reshape(28,28),vmin=0,vmax=255,cmap="gray",interpolation="nearest")
            ax.set_title(f"{writer}/{index} actual {text}, pred {pred}\n{v.name}",fontsize=8)
            ax.axis("off")
    fig.suptitle("EXP-CMB-003: real held-out handwriting, frozen writer-specific MLP",fontsize=11)
    fig.tight_layout()
    fig.savefig(figdir/"transform_examples.png",dpi=180)
    plt.close(fig)


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()
    out=args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    figdir=out/"figures"
    figdir.mkdir(parents=True,exist_ok=True)
    self_test()
    started=time.perf_counter()
    manifest,prior,table,raw,labels=core.load_inputs()
    models=core.load_models(manifest)
    normal=models["normal_mlp.joblib"]
    X=np.concatenate([raw[w].reshape(640,784).astype(np.float32) for w in WRITERS])
    y=np.concatenate([labels[w] for w in WRITERS]).astype(int)
    writers=np.repeat(np.asarray(WRITERS),640)
    indices=np.tile(np.arange(640),4)
    assert len(X)==2560 and len(prior)==2560
    assert np.array_equal(writers,[r["writer"] for r in prior])
    assert np.array_equal(indices,[r["index_in_writer"] for r in prior])
    assert np.array_equal(y,[LABELS.index(r["true_label"]) for r in prior])
    maskw={w:(writers==w) for w in WRITERS}

    official_x,official_y=load_official(manifest)
    models_all={"Normal":normal,"Combined":None}
    base_pred={}
    base_pred["Normal"]=normal.predict(X).astype(int)
    com_pred=np.zeros(2560,dtype=int)
    for w in WRITERS:
        com_pred[maskw[w]]=models["combined_heldout_"+str(w)+"_mlp.joblib"].predict(X[maskw[w]])
    base_pred["Combined"]=com_pred
    for model,col in (("Normal","normal_pred"),("Combined","combined_pred")):
        expected=np.asarray([LABELS.index(r[col]) for r in prior])
        assert np.array_equal(base_pred[model],expected),(model,"baseline mismatch")
    assert (np.sum(base_pred["Normal"]==y),np.sum(base_pred["Combined"]==y))==(1602,2080)
    print("REPRODUCED ALL ORIGINAL OOF PREDICTIONS FOR NORMAL AND COMBINED",flush=True)

    base_official={"Normal":normal.predict(official_x).astype(int)}
    for w in WRITERS:
        base_official["Combined_"+str(w)]=models["combined_heldout_"+str(w)+"_mlp.joblib"].predict(official_x).astype(int)
    assert np.mean(base_official["Normal"]==official_y)==manifest["evaluations"]["normal_official_dev_validation"]["accuracy"]
    original_mass=np.maximum(X.sum(axis=1),1)
    original_official_mass=np.maximum(official_x.sum(axis=1),1)

    rows_oof=[];rows_writer=[];rows_group=[];rows_class=[];rows_pair=[];rows_official=[]
    predrows=[];diagnostics=[];full_oof={}
    for cond in VARIANTS:
        transformed=transform(X,cond)
        x_off=transform(official_x,cond)
        mass_loss=np.maximum(0,1-transformed.sum(axis=1)/original_mass)
        mass_change=(transformed.sum(axis=1)-X.sum(axis=1))/original_mass
        clip_or_change=float(np.mean(mass_loss>1e-8))
        s_time=time.perf_counter()
        new_pred={"Normal":normal.predict(transformed).astype(int)}
        cp=np.zeros_like(y)
        for w in WRITERS:
            cp[maskw[w]]=models["combined_heldout_"+str(w)+"_mlp.joblib"].predict(transformed[maskw[w]]).astype(int)
        new_pred["Combined"]=cp
        handmade_predict_sec=time.perf_counter()-s_time
        full_oof[cond.name]=cp.copy()
        # Ensure first condition exactly reconstructs original and no label-based changes.
        if cond.name=="original":
            assert all(np.array_equal(new_pred[m],base_pred[m]) for m in ("Normal","Combined"))

        diagnostics.append({"variant":cond.name,"family":cond.family,
           "handmade_any_mass_loss_fraction":clip_or_change,
           "handmade_average_fraction_mass_lost":float(mass_loss.mean()),
           "handmade_average_fraction_mass_change":float(mass_change.mean()),
           "handmade_fraction_nonidentical":float(np.mean(np.any(transformed!=X,axis=1))),
           "official_average_fraction_mass_lost":float(np.maximum(0,1-x_off.sum(1)/original_official_mass).mean()),
           "handmade_model_prediction_seconds":handmade_predict_sec})
        for m in ("Normal","Combined"):
            p=new_pred[m]
            before=base_pred[m]
            base=metrics(y,before)
            now=metrics(y,p)
            paired=paired_counts(y,before,p)
            rows_oof.append({"variant":cond.name,"family":cond.family,"model":m,
               **{k+"_baseline":v for k,v in base.items() if k in ("accuracy","macro_f1")},
               **now,"delta_acc_pp":100*(now["accuracy"]-base["accuracy"]),
               "delta_macro_f1":now["macro_f1"]-base["macro_f1"],
               **paired})
            for w in WRITERS:
                z=maskw[w]
                mm=metrics(y[z],p[z])
                pp=paired_counts(y[z],before[z],p[z])
                rows_writer.append({"variant":cond.name,"model":m,"writer":w,
                  **mm,"delta_acc_pp":100*(mm["accuracy"]-np.mean(before[z]==y[z])),
                  **pp})
            for name,group_mask in (("digit",y<10),("letter",y>=10)):
                mm=metrics(y[group_mask],p[group_mask],classes=(np.arange(10) if name=="digit" else np.arange(10,16)))
                pp=paired_counts(y[group_mask],before[group_mask],p[group_mask])
                rows_group.append({"variant":cond.name,"model":m,"group":name,
                  **mm,"delta_acc_pp":100*(mm["accuracy"]-np.mean(before[group_mask]==y[group_mask])),
                  **pp})
            prec,rec,f1,support=precision_recall_fscore_support(y,p,labels=np.arange(16),zero_division=0)
            for j,c in enumerate(LABELS):
                z=y==j
                rows_class.append({"variant":cond.name,"model":m,"class":c,"n":int(support[j]),
                  "precision":float(prec[j]),"recall":float(rec[j]),"f1":float(f1[j]),
                  "before_correct":int(np.sum(before[z]==y[z])),
                  "after_correct":int(np.sum(p[z]==y[z])),
                  "corrections":int(np.sum((before[z]!=y[z])&(p[z]==y[z]))),
                  "induced_errors":int(np.sum((before[z]==y[z])&(p[z]!=y[z])))})
            for actual,rival in FOCUS:
                mask=y==LABELS.index(actual)
                wrong=int(np.sum(p[mask]==LABELS.index(rival)))
                old=int(np.sum(before[mask]==LABELS.index(rival)))
                rows_pair.append({"variant":cond.name,"model":m,"writer":"all",
                  "actual":actual,"predicted":rival,"original_errors":old,
                  "new_errors":wrong,"delta_wrong":wrong-old})
                for w in WRITERS:
                    z=mask&maskw[w]
                    rows_pair.append({"variant":cond.name,"model":m,"writer":w,
                     "actual":actual,"predicted":rival,
                     "original_errors":int(np.sum(before[z]==LABELS.index(rival))),
                     "new_errors":int(np.sum(p[z]==LABELS.index(rival))),
                     "delta_wrong":int(np.sum(p[z]==LABELS.index(rival))-
                                      np.sum(before[z]==LABELS.index(rival)))})
            # Save image-level paired transitions as compressed CSV
            for i in range(len(y)):
                predrows.append((cond.name,m,int(writers[i]),int(indices[i]),LABELS[y[i]],
                   LABELS[before[i]],LABELS[p[i]],int(before[i]==y[i]),int(p[i]==y[i])))

        for key,model in (("Normal",normal),*[( "Combined_"+str(w),
            models["combined_heldout_"+str(w)+"_mlp.joblib"]) for w in WRITERS]):
            t0=time.perf_counter()
            q=model.predict(x_off).astype(int)
            tm=time.perf_counter()-t0
            before=base_official[key]
            mm=metrics(official_y,q)
            pp=paired_counts(official_y,before,q)
            rows_official.append({"variant":cond.name,"model":key,
                **mm,"delta_acc_pp":100*(mm["accuracy"]-np.mean(before==official_y)),
                "prediction_seconds":tm,**pp})
        plot_cm(y,new_pred["Combined"],figdir,cond.name)
        print(cond.name,"| Combined",rows_oof[-1]["accuracy"],
              "| net",rows_oof[-1]["net_correct"],flush=True)

    # FDR for 10 perturbations within the same model; avoid treating baseline as a hypothesis.
    for m in ("Normal","Combined"):
        rr=[r for r in rows_oof if r["model"]==m and r["variant"]!="original"]
        bh(rr)
    # Forward margins of true class vs predeclared competitor on same held-out images.
    margins=[]
    for cond in VARIANTS:
        x=transform(X,cond)
        lnorm=core.logits(normal,x)
        lcom=np.empty((2560,16),dtype=float)
        for w in WRITERS:
            z=maskw[w]
            lcom[z]=core.logits(models["combined_heldout_"+str(w)+"_mlp.joblib"],x[z])
        for m,log in (("Normal",lnorm),("Combined",lcom)):
            for wstr,actual,rival in PRIMARY:
                w=int(wstr)
                z=maskw[w]&(y==LABELS.index(actual))
                aa=log[z,LABELS.index(actual)]-log[z,LABELS.index(rival)]
                oldx=core.logits(normal if m=="Normal" else models["combined_heldout_"+str(w)+"_mlp.joblib"],X[z])
                old=oldx[:,LABELS.index(actual)]-oldx[:,LABELS.index(rival)]
                nwrong=int(np.sum(full_oof[cond.name][z]==LABELS.index(rival))) if m=="Combined" else ""
                margins.append({"variant":cond.name,"model":m,"writer":w,"actual":actual,
                    "competitor":rival,"n":int(z.sum()),
                    "mean_margin":float(aa.mean()),"mean_delta_margin":float((aa-old).mean()),
                    "fraction_margin_below_zero":float(np.mean(aa<0)),
                    "target_pair_error_count":nwrong})

    # Explicit followup gate: no new trained pipeline can be claimed by this diagnostic.
    crows=[r for r in rows_oof if r["model"]=="Combined" and r["variant"]!="original"]
    official_base={w:np.mean(base_official["Combined_"+str(w)]==official_y) for w in WRITERS}
    candidates=[]
    for r in crows:
        v=r["variant"]
        wr=[a for a in rows_writer if a["variant"]==v and a["model"]=="Combined"]
        os=[a for a in rows_official if a["variant"]==v and a["model"].startswith("Combined_")]
        good=(r["BH_q_10_conditions"]<.05 and r["net_correct"]>=13
              and sum(a["net_correct"]>=0 for a in wr)>=3
              and min(a["delta_acc_pp"] for a in os)>=-.3)
        if good:
            candidates.append({"variant":v,"oof_delta_pp":r["delta_acc_pp"],
                               "BH_q":r["BH_q_10_conditions"],
                               "min_official_delta_pp":min(a["delta_acc_pp"] for a in os),
                               "writers_not_worse":sum(a["net_correct"]>=0 for a in wr)})
    core.write_csv(out/"oof_metrics.csv",rows_oof)
    core.write_csv(out/"writer_metrics.csv",rows_writer)
    core.write_csv(out/"digit_letter_metrics.csv",rows_group)
    core.write_csv(out/"per_class_metrics.csv",rows_class)
    core.write_csv(out/"pair_confusion_changes.csv",rows_pair)
    core.write_csv(out/"official_validation_metrics.csv",rows_official)
    core.write_csv(out/"focus_margin_diagnostics.csv",margins)
    core.write_csv(out/"transform_diagnostics.csv",diagnostics)
    with gzip.open(out/"paired_predictions.csv.gz","wt",newline="",encoding="utf-8") as f:
        wr=csv.writer(f)
        wr.writerow(["variant","model","writer","index_in_writer","actual","original_prediction",
                     "transformed_prediction","original_correct","transformed_correct"])
        wr.writerows(predrows)
    plot_examples(raw,labels,models,figdir)
    plot_tradeoff(rows_oof,figdir)
    report={"experiment_id":"EXP-CMB-003","status":"completed_input_diagnostic",
       "baseline_reproduced":{"normal_correct":1602,"combined_correct":2080,"n":2560},
       "hypothesis":"B spatial and ink differences impact held-out writer errors",
       "conditions":[{"name":v.name,"family":v.family,"value":v.value} for v in VARIANTS],
       "predefined_candidate_gate":{
         "accuracy_gain_min_pp":.5,"paired_BH_q_lt":.05,
         "writer_nonworsening_min":3,"per_fold_official_delta_acc_min_pp":-.3},
       "screening_candidates_for_retraining":candidates,
       "decision":"SCREEN_RETRAIN_CANDIDATES" if candidates else "NO_GLOBAL_RETRAIN_CANDIDATE_FROM_TESTED_VARIANTS",
       "note":"Transform of validation alone is a fixed-model diagnostic; no inference accuracy gain proves retrained model improvement.",
       "normal_comparison":True,"normal_and_combined_frozen_model_count":5,
       "new_training_runs":0,"official_original_test_used":False,"professor_final_test_used":False,
       "versions":{"sklearn":sklearn.__version__,"numpy":np.__version__,"scipy":scipy.__version__},
       "models_zip_members_sha256":manifest["models_zip_members_sha256"],
       "elapsed_seconds":time.perf_counter()-started}
    (out/"run_manifest.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    md=["# EXP-CMB-003 — Frozen Combined MLP input sensitivity: automatic metrics","",
        "**Status:** experimentally executed; final research interpretation in EXP유003 log.",
        "Frozen Normal 1 + Combined 4, original 2,560 Hand-made predictions revalidated.",
        "Official Original Test/Professor Test not loaded, no training.",
        "",
        "| Condition | OOF Accuracy | Δpp | Corrected | Newly wrong | Flip rate | McNemar exact p | BH q (10) |",
        "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for r in rows_oof:
        if r["model"]=="Combined":
            md.append("| {} | {:.3%} | {:+.3f} | {} | {} | {:.2%} | {:.4g} | {} |".format(
                r["variant"],r["accuracy"],r["delta_acc_pp"],r["corrections"],
                r["induced_errors"],r["prediction_flip_rate"],r["mcnemar_exact_p"],
                ("{:.4g}".format(r["BH_q_10_conditions"]) if "BH_q_10_conditions" in r else "—")))
    md+=["",f"**Gate result:** {report['decision']}","",
         "Screening candidates are not finished models; further 4-Fold Training/Validation evaluation would be a separate study.",
         "No criterion met does not imply no geometry effect: it means these global single-variable inference-time interventions are not a ready-to-deploy solution.",
         "Consult writer_metrics.csv, pair_confusion_changes.csv, focus_margin_diagnostics.csv and official_validation_metrics.csv.",
         ""]
    (out/"SUMMARY.md").write_text("\n".join(md),encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)
    print("EXP-CMB-003 COMPLETE: 11 CONDITIONS / 5 FROZEN MODELS / NO TEST",flush=True)


if __name__=="__main__":
    main()


## render_cm_viridis

원본: [05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py)  
Git blob: `c233e0fead21ab2fe3f0977c0c0d1a4f4596d1fb`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-001: regenerate only approved four Viridis OOF confusion matrices.

Run in repository root:
 python 05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py
or:
 python 05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py      --predictions path/to/OOF_predictions.csv --output-dir path/to/figures

No model training. No edits to the source OOF_predictions.csv.

CLI 인자 (정적 추출): `--predictions`, `--output-dir`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py
#!/usr/bin/env python3
"""EXP-CMB-001: regenerate only approved four Viridis OOF confusion matrices.

Run in repository root:
 python 05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py
or:
 python 05_model_optimize/MLP/Combined/scripts/render_cm_viridis.py \
     --predictions path/to/OOF_predictions.csv --output-dir path/to/figures

No model training. No edits to the source OOF_predictions.csv.
"""
from __future__ import annotations
import argparse
import csv
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
LABELS=tuple('0123456789ABCDEF')
N_CLASSES=16
ROW_COUNT=160
COLOR_SCALE='viridis'
COLOR_MIN=0
COUNT_COLOR_MAX=160
NORM_COLOR_MAX=100
plt.rcParams.update({
 'font.family':'DejaVu Sans', 'figure.facecolor':'white', 'axes.facecolor':'white',
 'savefig.facecolor':'white', 'font.size':9,'axes.labelsize':11,
 'axes.titlesize':13,'axes.titleweight':'semibold',
 'xtick.labelsize':9,'ytick.labelsize':9,
})
def luminance(rgb):
    r, g, b = rgb[:3]
    return 0.2126*r + 0.7152*g + 0.0722*b


def draw_heatmap(ax, cm, kind, title, show_ylabel=True):
    if kind == 'count':
        matrix = cm.astype(float)
        maximum, textfmt = COUNT_COLOR_MAX, lambda v: str(int(v))
    else:
        matrix = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        maximum, textfmt = NORM_COLOR_MAX, lambda v: f'{v:.1f}'
    norm = Normalize(vmin=COLOR_MIN, vmax=maximum)
    im = ax.imshow(matrix, cmap=COLOR_SCALE, norm=norm, interpolation='nearest', aspect='equal')
    ax.set_xticks(np.arange(N_CLASSES), labels=LABELS)
    ax.set_yticks(np.arange(N_CLASSES), labels=LABELS)
    ax.tick_params(axis='both', which='major', length=0, pad=6)
    ax.set_xlabel('Predicted label', labelpad=10)
    ax.set_ylabel('True label' if show_ylabel else '', labelpad=9)
    ax.set_title(title, pad=14)

    # A subtle square grid improves locating the row/column while keeping the original colors.
    ax.set_xticks(np.arange(-.5, N_CLASSES, 1), minor=True)
    ax.set_yticks(np.arange(-.5, N_CLASSES, 1), minor=True)
    ax.grid(which='minor', color='white', linestyle='-', linewidth=.38, alpha=.31)
    ax.tick_params(which='minor', bottom=False, left=False)
    for side in ('left', 'right', 'top', 'bottom'):
        ax.spines[side].set_visible(False)

    cmap = matplotlib.colormaps[COLOR_SCALE]
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            if cm[i, j] == 0:
                continue  # Important: suppress only zero values, never true counts.
            value = matrix[i, j]
            rgba = cmap(norm(value))
            fg = '#14202e' if luminance(rgba) > .53 else '#ffffff'
            ax.text(j, i, textfmt(value), ha='center', va='center', fontsize=6.15,
                    weight='medium', color=fg)
    return im


def add_colorbar(fig, im, ax, kind, horizontal=False):
    kw = {'ax': ax, 'fraction': .045, 'pad': .026, 'shrink': .89}
    cb = fig.colorbar(im, **kw)
    cb.set_label('Number of images' if kind == 'count' else 'Percentage of true class (%)',
                 rotation=90, labelpad=10, fontsize=9)
    cb.outline.set_visible(False)
    return cb


def save(fig, stem: Path, *, dpi=300):
    fig.savefig(stem.with_suffix('.png'), dpi=dpi, facecolor='white', bbox_inches='tight', pad_inches=.14)
    plt.close(fig)




def load_cm_from_predictions(path:Path,model:str)->np.ndarray:
    with path.open(encoding='utf-8-sig',newline='') as f:
        data=list(csv.DictReader(f))
    if len(data)!=2560:raise ValueError(f'Expected 2560 predictions, got {len(data)}')
    col={'Normal':'normal_pred','Combined':'combined_pred'}[model]
    true=np.array([LABELS.index(r['true_label']) for r in data],dtype=np.int64)
    pred=np.array([LABELS.index(r[col]) for r in data],dtype=np.int64)
    cm=np.zeros((16,16),dtype=np.int64)
    np.add.at(cm,(true,pred),1)
    if not np.array_equal(cm.sum(axis=1),np.full(16,ROW_COUNT)):
        raise ValueError('Expected 160 true samples per class')
    return cm


def main():
    here=Path(__file__).resolve().parents[1]
    default=here/'results'/'EXP_CMB_001'
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--predictions',type=Path,default=default/'OOF_predictions.csv')
    ap.add_argument('--output-dir',type=Path,default=default/'figures')
    args=ap.parse_args()
    args.output_dir.mkdir(parents=True,exist_ok=True)
    for model in ('Normal','Combined'):
        cm=load_cm_from_predictions(args.predictions,model)
        for kind in ('count','normalized'):
            fig,ax=plt.subplots(figsize=(10,9),constrained_layout=True)
            title=f'MLP | {model} | Hand-made OOF'+(' | Row-normalized (%)' if kind=='normalized' else '')
            im=draw_heatmap(ax,cm,kind,title)
            add_colorbar(fig,im,ax,kind)
            save(fig,args.output_dir/f'{model.lower()}_{kind}_viridis',dpi=300)
    print('Rendered exactly four Viridis CM PNG files, without retraining.')

if __name__=='__main__':main()

## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")